## Per il NER genero un dataset sintetico. Sarà un caso non realistico ma solo a scopo didattico. Un vero NER su un dataset relae sraà allenato nella lezione sui transformers

In [3]:
import random

# --- 1. GENERATORE DI DATE (200+ variazioni) ---
def get_diverse_dates(n=200):
    days = ["lunedì", "martedì", "mercoledì", "giovedì", "venerdì", "sabato", "domenica"]
    months = ["gennaio", "febbraio", "marzo", "aprile", "maggio", "giugno",
              "luglio", "agosto", "settembre", "ottobre", "novembre", "dicembre"]
    relatives = ["oggi", "domani", "dopodomani", "ieri", "l'altro ieri", "presto"]
    years = [str(y) for y in range(1990, 2030)]

    generated_dates = []

    for _ in range(n):
        # Scegliamo a caso uno stile di data
        style = random.choice([1, 2, 3, 4, 5])

        if style == 1: # Es: "12 marzo"
            d = f"{random.randint(1, 31)} {random.choice(months)}"
        elif style == 2: # Es: "venerdì prossimo"
            d = f"{random.choice(days)} prossimo"
        elif style == 3: # Es: "il 15/05/2024"
            d = f"{random.randint(1, 28)}/{random.randint(1, 12)}/{random.choice(years)}"
        elif style == 4: # Es: "domani" (singola parola)
            d = random.choice(relatives)
        elif style == 5: # Es: "gennaio 2025"
            d = f"{random.choice(months)} {random.choice(years)}"

        generated_dates.append(d)

    return generated_dates

# --- 2. GENERATORE DI TEMPLATE (200+ variazioni) ---
def get_diverse_templates(n=200):
    # Struttura: [Soggetto/Verbo] + [Preposizione/Contesto] + {date} + [Finale opzionale]

    starts = [
        "La riunione è fissata per", "Ci vediamo", "Il pacco arriva", "Sono nato",
        "La scadenza è", "Non sarò disponibile", "Ho prenotato il volo per",
        "L'evento inizia", "Il concerto si terrà", "Pagamento ricevuto",
        "Ricordati di chiamarmi", "L'esame sarà", "Siamo partiti", "Tornerò",
        "La festa è stata spostata a", "Consegna prevista per", "Il contratto scade",
        "Abbiamo appuntamento", "Il negozio chiude", "L'offerta è valida fino a"
    ]

    endings = [
        "", ".", " alle 15:00.", " puntuali.", " senza ritardi.", " di mattina.",
        " o forse dopo.", ", spero tu possa venire.", ", non mancare!", " circa."
    ]

    templates = []
    for _ in range(n):
        s = random.choice(starts)
        e = random.choice(endings)
        # Il placeholder {date} verrà sostituito dopo
        templates.append(f"{s} {{date}}{e}")

    return templates

# Stampiamo 3 esempi per vedere il risultato
print(f"Dataset generato: {len(data)} frasi.\n")
for i in range(3):
    print(f"Frase: {data[i][0]}")
    print(f"Label: {data[i][1]}")
    print("-" * 30)

Dataset generato: 200 frasi.

Frase: ['Non', 'sarò', 'disponibile', "l'altro", 'ieri', 'senza', 'ritardi', '.']
Label: [0, 0, 0, 1, 1, 0, 0, 0]
------------------------------
Frase: ['Sono', 'nato', '6/8/2015', 'circa', '.']
Label: [0, 0, 1, 0, 0]
------------------------------
Frase: ['La', 'scadenza', 'è', 'domani', '.']
Label: [0, 0, 0, 1, 0]
------------------------------


In [40]:
# --- 3. CREAZIONE DATASET BINARIO ---
def create_binary_dataset(n_samples=200):
    dates = get_diverse_dates(n_samples)
    templates = get_diverse_templates(n_samples)
    dataset = []

    for i in range(n_samples):
        date_str = dates[i]
        template = templates[i]

        # Uniamo template e data
        sentence = template.format(date=date_str)

        # Tokenizzazione semplice (split spazi e punteggiatura basilare)
        # Nota: per un caso reale servirebbe una tokenizzazione più robusta
        sentence_clean = sentence.replace(".", " .").replace(",", " ,").lower()
        tokens = sentence_clean.split()

        # Creazione Label Binarie
        # 0 = parola normale, 1 = parte della data
        labels = []

        # Creiamo un set delle parole che compongono la data corrente per un controllo veloce
        # Es: date_str="12 marzo" -> date_tokens={"12", "marzo"}
        date_tokens_set = set(date_str.split())

        for token in tokens:
            # Check grezzo: se il token è presente nella stringa della data originale
            if token in date_tokens_set or token.strip("/.") in date_tokens_set:
                labels.append(1)
            else:
                labels.append(0)

        dataset.append((tokens, labels))

    return dataset

# --- ESECUZIONE ---
data = create_binary_dataset(14000)



In [42]:
data[:10]

[(['siamo', 'partiti', 'oggi', 'puntuali', '.'], [0, 0, 1, 0, 0]),
 (['il', 'contratto', 'scade', '22', 'luglio', '.'], [0, 0, 0, 1, 1, 0]),
 (['il', 'concerto', 'si', 'terrà', '13/1/1992', 'alle', '15:00', '.'],
  [0, 0, 0, 0, 1, 0, 0, 0]),
 (["l'evento", 'inizia', '10/8/2019', 'puntuali', '.'], [0, 0, 1, 0, 0]),
 (['la',
   'scadenza',
   'è',
   '17/4/1994',
   ',',
   'spero',
   'tu',
   'possa',
   'venire',
   '.'],
  [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]),
 (["l'evento", 'inizia', '21/10/2015', 'o', 'forse', 'dopo', '.'],
  [0, 0, 1, 0, 0, 0, 0]),
 (['sono', 'nato', 'lunedì', 'prossimo', 'alle', '15:00', '.'],
  [0, 0, 1, 1, 0, 0, 0]),
 (['non', 'sarò', 'disponibile', '21', 'aprile', 'di', 'mattina', '.'],
  [0, 0, 0, 1, 1, 0, 0, 0]),
 (['sono', 'nato', '7', 'settembre', '.'], [0, 0, 1, 1, 0]),
 (['sono', 'nato', 'oggi', 'o', 'forse', 'dopo', '.'], [0, 0, 1, 0, 0, 0, 0])]

### Il dataset deve essere composto da coppie (x,y) in cui la x è la lista di tokens o parole e la y ha la stessa lunghezza ed è composta da 0 o 1 a seconda che il token sia relativo ad una data o meno

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Costruiamo il vocabolario (Mapping Parola -> Indice)
# Usiamo i dati generati nel passo precedente (variabile 'data')
all_tokens = set()
for tokens, _ in data:
    for token in tokens:
        all_tokens.add(token)

vocab = {word: i + 1 for i, word in enumerate(all_tokens)} # +1 perché 0 serve per il padding
vocab['<PAD>'] = 0
vocab_size = len(vocab)

# 2. Convertiamo Frasi e Label in numeri e facciamo Padding
max_len = max(len(x[0]) for x in data) # Lunghezza massima nel dataset

X_numerized = []
y_numerized = []

for tokens, labels in data:
    # Converti parole in indici
    x_seq = [vocab[token] for token in tokens]
    # Padding: riempiamo con 0 fino a max_len
    x_seq += [0] * (max_len - len(x_seq))
    labels += [0] * (max_len - len(labels)) # Padding anche per le label )

    X_numerized.append(x_seq)
    y_numerized.append(labels)

# Convertiamo in Tensori PyTorch
X_tensor = torch.LongTensor(X_numerized)
y_tensor = torch.FloatTensor(y_numerized) # Float perché serve per la Loss Binaria

# 3. Split Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

# Dataloader per gestire i batch
batch_size = 16
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)

print(f"Vocab Size: {vocab_size}")
print(f"Max Len: {max_len}")
print(f"Train shape: {X_train.shape}")

Vocab Size: 2642
Max Len: 14
Train shape: torch.Size([11200, 14])


### Definiamo la rete: un bidirectional RNN

In [43]:
class BiRNN_NER(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(BiRNN_NER, self).__init__()

        # Embedding: trasforma indici interi in vettori densi
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # LSTM Bidirezionale
        # batch_first=True significa che l'input è (batch, seq, features)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)

        # Fully Connected Layer
        # hidden_dim * 2 perché abbiamo due direzioni (avanti + indietro)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len]

        embeds = self.embedding(x)
        # embeds shape: [batch_size, seq_len, embedding_dim]

        lstm_out, _ = self.lstm(embeds)
        # lstm_out shape: [batch_size, seq_len, hidden_dim * 2]

        output = self.fc(lstm_out)
        # output shape: [batch_size, seq_len, 1]

        return output.squeeze(-1) # Rimuoviamo l'ultima dimensione inutile

### Inizializziamo il modello e l'optimizer

In [44]:
# Iperparametri
EMBEDDING_DIM = 32
HIDDEN_DIM = 64
EPOCHS = 10
LEARNING_RATE = 0.001

model = BiRNN_NER(vocab_size, EMBEDDING_DIM, HIDDEN_DIM)

# Binary Cross Entropy con Logits (più stabile numericamente della Sigmoid esplicita)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


In [45]:
def calculate_exact_match_accuracy(preds, labels, inputs):
    # preds: tensore con probabilità (logits)
    # labels: tensore con 0 e 1
    # inputs: ci serve per capire dove c'è il padding (valore 0) e ignorarlo

    # Applichiamo sigmoide e arrotondiamo a 0 o 1
    predicted_classes = (torch.sigmoid(preds) > 0.5).float()

    correct_sequences = 0
    total_sequences = labels.size(0)

    for i in range(total_sequences):
        # Troviamo la lunghezza reale della frase (ignorando il padding)
        # Il padding nell'input è rappresentato da 0
        real_len = (inputs[i] != 0).sum().item()

        # Confrontiamo solo i token reali, non il padding
        pred_seq = predicted_classes[i][:real_len]
        true_seq = labels[i][:real_len]

        # Se tutti gli elementi sono uguali
        if torch.equal(pred_seq, true_seq):
            correct_sequences += 1

    return correct_sequences / total_sequences

### Ciclo di training

In [46]:
# --- CICLO DI TRAINING ---
print("Inizio Training...")
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = model(x_batch)

        # Attenzione: La loss calcola anche il padding, il che sporca un po' il gradiente
        # ma per un toy dataset funziona comunque bene.
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation sul Test Set ad ogni epoca
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test)
        acc = calculate_exact_match_accuracy(test_preds, y_test, X_test)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Test Exact Acc: {acc*100:.2f}%")

Inizio Training...
Epoch 1/3 | Loss: 0.0653 | Test Exact Acc: 99.36%
Epoch 2/3 | Loss: 0.0014 | Test Exact Acc: 99.89%
Epoch 3/3 | Loss: 0.0003 | Test Exact Acc: 100.00%


In [26]:
# --- TEST FINALE ---
print("\n--- Esempio di Predizione ---")
idx = 0 # Prendiamo il primo elemento del test set
input_sample = X_test[idx].unsqueeze(0)
true_label = y_test[idx]

model.eval()
with torch.no_grad():
    logit_pred = model(input_sample)
    prob_pred = torch.sigmoid(logit_pred).squeeze()
    class_pred = (prob_pred > 0.5).int()

# Ricostruiamo la frase per leggere meglio
# Creiamo un dizionario inverso: Indice -> Parola
idx2word = {v: k for k, v in vocab.items()}

real_len = (input_sample[0] != 0).sum().item()
words = [idx2word[idx.item()] for idx in input_sample[0][:real_len]]
pred_tags = class_pred[:real_len].tolist()
true_tags = true_label[:real_len].int().tolist()

print(f"Frase: {' '.join(words)}")
print(f"Reale: {true_tags}")
print(f"Pred.: {pred_tags}")

if pred_tags == true_tags:
    print("RISULTATO: Giusto")
else:
    print("RISULTATO: Errore")


--- Esempio di Predizione ---
Frase: Pagamento ricevuto agosto 1997 senza ritardi .
Reale: [0, 0, 1, 1, 0, 0, 0]
Pred.: [0, 0, 1, 1, 0, 0, 0]
RISULTATO: Giusto


In [50]:
def predict_custom_sentence(sentence, model, vocab):
    model.eval() # Modalità valutazione

    # 1. Preprocessing identico al training
    # Stacchiamo la punteggiatura per coerenza con il tokenizer usato prima
    sentence_clean = sentence.replace(".", " .").replace(",", " ,").replace("?", " ?").lower()
    tokens = sentence_clean.split()

    # 2. Conversione in indici (Numericalization)
    ids = []
    unknown_words = []

    for token in tokens:
        if token in vocab:
            ids.append(vocab[token])
        else:
            # Se la parola non è nel vocabolario, usiamo 0 (Padding)
            ids.append(0)
            unknown_words.append(token)

    if unknown_words:
        print(f"Parole ignorate (non viste nel training): {unknown_words}")

    # 3. Creazione Tensore (Batch size = 1)
    input_tensor = torch.LongTensor(ids).unsqueeze(0) # Shape: [1, seq_len]

    # 4. Predizione
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).int() # Soglia a 0.5

    # Rimuoviamo la dimensione del batch per leggere i risultati
    preds_list = preds.squeeze().tolist()

    # Gestione caso singola parola (tolist restituisce int invece di list)
    if not isinstance(preds_list, list):
        preds_list = [preds_list]

    # 5. Visualizzazione Risultati
    print("\n" + "="*30)
    print(f"Frase: '{sentence}'")
    print("-" * 30)

    found_entity = False
    output_str = ""

    for word, tag in zip(tokens, preds_list):
        if tag == 1:
            output_str += f"[{word.upper()}] "
            found_entity = True
        else:
            output_str += f"{word} "

    print(f"Risultato: {output_str}")
    print("="*30 + "\n")


predict_custom_sentence("ci vediamo lunedì  prossimo alle 21", model, vocab)


Parole ignorate (non viste nel training): ['ci']

Frase: 'ci vediamo lunedì  prossimo alle 21'
------------------------------
Risultato: ci vediamo [LUNEDÌ] [PROSSIMO] alle 21 



### Un vero NER avrebbe una label per l'inzio dell'entità per esempio B-DATE e una label per la continuazione delle entità I-DATE in modo tale da mergiare token che afferiscono alla stessa entità